In [2]:
import os
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import scanpy as sc
import anndata
import timm
from huggingface_hub import login
import torch

In [3]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field


class QueryConfig(BaseModel):
    # If csv_path is None, the script will try to load metadata from the HEST dataset on HuggingFace.
    csv_path: Optional[Path] = None
    column: str
    method: Literal["startswith", "contains", "equals"] = "startswith"
    pattern: str
    # optional exact-match filters: {"disease_state": "Healthy"}
    filters: Optional[dict] = None


class PreprocessConfig(BaseModel):
    # filesystem
    data_path: Path
    tif_path: Optional[Path] = None
    st_path: Optional[Path] = None
    metadata_path: Optional[Path] = None
    save_path: Optional[Path] = None

    # sample selection: either a direct list/file or a query against the metadata CSV
    selection_mode: Literal["list", "query"] = "list"
    # list mode
    ids_to_query: Optional[List[str]] = None
    ids_file: Optional[Path] = None
    # query mode
    ids_query: Optional[QueryConfig] = None

    # runtime
    device: str = "cuda"

    # embedding options
    models_to_run: List[str] = Field(default_factory=lambda: ["conch", "uni"])  # supported: conch, uni
    save_augmented: bool = True

    # gene selection (optional)
    run_gene_selection: bool = False
    hvg_top_k: int = 0           # size of final HVG list; 0 disables HVG output
    hmhvg_top_k: int = 0         # size of final HMHVG list; 0 disables HMHVG output
    deg_top_k: int = 0           # size per-group for DEG selection; 0 disables DEG output
    deg_groupby: Optional[str] = None  # obs column to group by for DEG selection
    deg_groups: Optional[List[str]] = None  # optional specific groups to test; defaults to all
    deg_label_dir: Optional[Path] = Path("data/ST-pat/lbl")  # optional: directory with per-slide region labels
    deg_subseries_column: str = "subseries"  # metadata column mapping id -> subseries (e.g., A1, B1)
    gene_list_filename: str = "selected_gene_list.txt"  # customize output filename for selected genes

    # error handling
    halt_on_error: bool = False  # If True, stop immediately on first error; if False, continue and collect failures
    
    # processing mode: 'both' (images + genes), 'images' (only image embeddings), 'genes' (only gene selection)
    process_mode: Literal["both", "images", "genes"] = "both"


def load_metadata_for_deg(cfg: PreprocessConfig) -> Optional[pd.DataFrame]:
    """Return metadata dataframe with id/subseries if available for DEG label mapping."""
    # Priority: explicit metadata_path; fallback to ids_query.csv_path if present
    meta_path = cfg.metadata_path or (cfg.ids_query.csv_path if cfg.selection_mode == "query" and cfg.ids_query and cfg.ids_query.csv_path else None)
    meta_df = pd.read_csv(meta_path)
    return meta_df

In [4]:
from utils.config_loader import load_toml_config

cfg_path = Path("configs/data_preprocessing/process_kidney.toml")
cfg = load_toml_config(cfg_path, PreprocessConfig, sections_to_flatten=["paths"])

meta_df = load_metadata_for_deg(cfg)

In [5]:
# disable pandas ... display options for cleaner notebook output
pd.set_option('display.max_columns', 1000)
meta_df.head()

,dataset_title,id,image_filename,organ,disease_state,oncotree_code,species,patient,st_technology,data_publication_date,license,study_link,download_page_link1,inter_spot_dist,spot_diameter,spots_under_tissue,preservation_method,nb_genes,treatment_comment,pixel_size_um_embedded,pixel_size_um_estimated,magnification,fullres_px_width,fullres_px_height,tissue,disease_comment,subseries,hest_version_added
0,Fresh Frozen Mouse Brain Hemisphere with 5K Mo...,TENX159,TENX159.tif,Brain,Healthy,NaN,Mus musculus,NaN,Xenium,7/31/24,Creative Commons Attribution,NaN,NaN,NaN,NaN,3429,Fresh Frozen,13780,NaN,0.273768,0.274027,40x,17051,24689,Brain,NaN,NaN,v1_1_0
1,FFPE Human Skin Primary Dermal Melanoma with 5...,TENX158,TENX158.tif,Skin,Cancer,SKCM,Homo sapiens,NaN,Xenium,7/31/24,Creative Commons Attribution,NaN,https://www.10xgenomics.com/datasets/xenium-pr...,NaN,NaN,2179,FFPE,10017,NaN,0.273777,0.273754,40x,18669,35787,Skin,Primary Dermal Melanoma,NaN,v1_1_0
2,FFPE Human Prostate Adenocarcinoma with 5K Hum...,TENX157,TENX157.tif,Prostate,Cancer,PRAD,Homo sapiens,NaN,Xenium,7/31/24,Creative Commons Attribution,NaN,https://www.10xgenomics.com/datasets/xenium-pr...,NaN,NaN,4427,FFPE,10006,NaN,0.273772,0.273741,40x,25002,49976,Prostate,NaN,NaN,v1_1_0
3,Characterization of immune cell populations in...,TENX156,TENX156.tif,Bowel,Cancer,COAD,Homo sapiens,Patient 1,Visium HD,7/11/24,Creative Commons Attribution,https://www.biorxiv.org/content/10.1101/2024.0...,https://www.10xgenomics.com/products/visium-hd...,2.0,2.0,2179,FFPE,18085,NaN,0.264583,0.273802,40x,71106,58791,Colon,Stage II-A,"Visium HD, Sample P1 CRC",v1_1_0
4,Characterization of immune cell populations in...,TENX155,TENX155.tif,Bowel,Cancer,COAD,Homo sapiens,Patient 1,Visium HD,7/11/24,Creative Commons Attribution,https://www.biorxiv.org/content/10.1101/2024.0...,https://www.10xgenomics.com/products/visium-hd...,2.0,2.0,2281,FFPE,18085,NaN,0.264583,0.273874,40x,75250,48740,Colon,NaN,"Visium HD, Sample P2 CRC",v1_1_0


In [6]:
meta_df['dataset_title'].unique()

array(['Fresh Frozen Mouse Brain Hemisphere with 5K Mouse Pan Tissue and Pathways Panel',
       'FFPE Human Skin Primary Dermal Melanoma with 5K Human Pan Tissue and Pathways Panel',
       'FFPE Human Prostate Adenocarcinoma with 5K Human Pan Tissue and Pathways Panel',
       'Characterization of immune cell populations in the tumor microenvironment of colorectal cancer using high definition spatial profiling',
       'Visium HD Spatial Gene Expression Library, Mouse Kidney (FFPE)',
       'Visium HD Spatial Gene Expression Library, Mouse Embryo (FFPE)',
       'Visium HD Spatial Gene Expression Library, Human Pancreas (FFPE)',
       'Preview Data: FFPE Human Lymph Node with 5K Pan Tissue and Pathways Panel',
       'Nextflow Pipeline for Visium and H&E Data from Patient-Derived Xenograft Samples',
       'Image-based spatial transcriptomics identifies molecular niche dysregulation associated with distal lung remodeling in pulmonary fibrosis',
       'Spatially resolved multiomics 

In [7]:
meta_df['st_technology'].value_counts()

st_technology
Visium                     602
Spatial Transcriptomics    552
Xenium                      65
Visium HD                   10
Name: count, dtype: int64

In [8]:
dfslice = meta_df[
    (meta_df['st_technology'] != 'Spatial Transcriptomics') & (meta_df['species'] == 'Homo sapiens')
]
len(dfslice['dataset_title'].unique())

88

In [9]:
dfslice['preservation_method'].value_counts()

preservation_method
Fresh Frozen    271
FFPE            139
PFA fixed         7
PFA               7
Name: count, dtype: int64

In [10]:
# display values for "organ" after uncapitalizing everything
dfslice['organ'] = dfslice['organ'].str.lower()
dfslice['organ'].value_counts()

/tmp/ipykernel_3569328/2784176621.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfslice['organ'] = dfslice['organ'].str.lower()


organ
bowel         84
skin          68
kidney        64
lung          59
brain         43
heart         43
prostate      36
breast        17
uterus        17
liver         16
pancreas       8
bladder        6
eye            6
lymphoid       5
cervix         5
lymph node     5
ovary          3
bone           1
Name: count, dtype: int64

In [11]:
# Load one st sample per dataset to inspect
from pathlib import Path
# define paths
data_path = Path("/storage/hest1k")   # local path to dataset
st_path = data_path / "st/"             # ST data path

# get one sample id per dataset title
sample_ids = dfslice.groupby('dataset_title')['id'].first().tolist()
assert len(sample_ids) == len(dfslice['dataset_title'].unique())

adata_lst = []
first = True
for fn in sample_ids:
    adata = anndata.read_h5ad(st_path / f"{fn}.h5ad")
    adata_lst.append(adata)
    if first:
        common_genes = adata.var_names 
        first = False
        print(fn, adata.shape)
        continue
    common_genes = set(common_genes).intersection(set(adata.var_names))
    print(fn, adata.shape, end="\t")

NCBI568 (2438, 36601)
NCBI603 (768, 33538)	NCBI691 (1388, 36601)	MISC32 (1928, 33538)	NCBI855 (1432, 17943)	MISC73 (3901, 19074)	TENX156 (2179, 18085)	

/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


ZEN49 (2203, 36601)	TENX138 (10641, 541)	TENX99 (23060, 541)	TENX97 (11845, 541)	TENX95 (11845, 541)	TENX139 (4167, 541)	TENX141 (4390, 541)	TENX142 (3743, 541)	TENX126 (2190, 541)	TENX140 (4513, 541)	TENX157 (4427, 10006)	TENX158 (2179, 10017)	TENX68 (1657, 18085)	NCBI595 (1303, 36601)	NCBI540 (413, 33538)	MEND54 (469, 33538)	NCBI785 (4180, 541)	NCBI776 (4992, 18085)	TENX134 (3811, 541)	TENX73 (10878, 18085)	TENX13 (3813, 33538)	TENX14 (4015, 33538)	TENX39 (2518, 17943)	TENX23 (4325, 1056)	TENX53 (4898, 36601)	TENX24 (4325, 36601)	TENX56 (4992, 1186)	TENX27 (4992, 36601)	TENX50 (2781, 17943)	TENX114 (6006, 541)	TENX70 (9080, 18085)	TENX28 (3138, 1142)	TENX29 (3138, 36601)	TENX30 (3468, 1253)	TENX31 (3468, 36601)	TENX15 (4247, 36601)	TENX119 (2263, 541)	TENX49 (2660, 17943)	TENX106 (2730, 541)	TENX71 (5936, 18085)	TENX121 (9200, 541)	TENX62 (3858, 18085)	TENX72 (6195, 18085)	TENX16 (4035, 36601)	TENX51 (3455, 17943)	TENX65 (4674, 18085)	TENX46 (3043, 17943)	TENX40 (4371, 17943)	TENX123

/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


TENX128 (2279, 18085)	TENX144 (1848, 18085)	MISC12 (4226, 33538)	

/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [12]:
def get_lower_cased_genes(var_names):
    return [gene.lower().replace("grch38______", "") for gene in var_names]

common_genes = set(get_lower_cased_genes(adata_lst[0].var_names))
for i, adata in enumerate(adata_lst):
    if i in [80]:
        print("Skipping problematic dataset at index", i)
        continue
    sample_genes = get_lower_cased_genes(adata.var_names)
    common_genes = set(common_genes).intersection(set(sample_genes))
    print(i, len(common_genes), f"\t{list(sample_genes)[:5]}", end="\n")
    if len(common_genes) < 1:
        print("Problematic dataset at index", i)
        break

0 36601 	['mir1302-2hg', 'fam138a', 'or4f5', 'al627309.1', 'al627309.3']


1 31915 	['mir1302-2hg', 'fam138a', 'or4f5', 'al627309.1', 'al627309.3']
2 31915 	['mir1302-2hg', 'fam138a', 'or4f5', 'al627309.1', 'al627309.3']
3 31915 	['mir1302-2hg', 'fam138a', 'or4f5', 'al627309.1', 'al627309.3']
4 17763 	['samd11', 'noc2l', 'klhl17', 'plekhn1', 'perm1']
5 17412 	['or4f5', 'samd11', 'noc2l', 'klhl17', 'plekhn1']
6 16867 	['samd11', 'noc2l', 'klhl17', 'plekhn1', 'perm1']
7 16867 	['mir1302-2hg', 'fam138a', 'or4f5', 'al627309.1', 'al627309.3']
8 450 	['lamp2', 's100b', 'sox9', 'cdk6', 'tuba1b']
9 105 	['runx1', 'itgax', 'tcim', 'lum', 'sec11c']
10 105 	['ldha', 'lum', 'nnmt', 'thy1', 'runx1']
11 105 	['tcim', 'runx1', 'sec11c', 'blank_0418', 'lum']
12 76 	['epcam', 'ccnd1', 'rnf43', 'cd47', 'stat1']
13 76 	['sdc1', 'angpt2', 'basp1', 'lpl', 'socs2']
14 76 	['ctnnb1', 'il13ra2', 'trem2', 's100b', 'notch2']
15 56 	['sfrp2', 'ctsk', 'mef2c', 'elf5', 'bank1']
16 56 	['ighg1', 'negcontrolcodeword_0538', 'igkc', 'unassignedcodeword_0461', 'irf8']
17 45 	['atp7a', 'camsap

In [13]:
gene_panel_size = pd.Series(
    [len(adata.var_names) for adata in adata_lst]
)

dfgenepanel = pd.DataFrame({
    'sample_id': sample_ids,
    'dataset_title': dfslice.groupby('dataset_title')['id'].first().index,
    'gene_panel_size': gene_panel_size
})

In [14]:
dfslice.groupby('dataset_title')['id'].first().index[2]

'A single-cell transcriptomic analysis of endometriosis'

In [15]:
gene_panel_size.describe()

count       88.000000
mean     18749.954545
std      14043.579025
min        538.000000
25%        927.250000
50%      18085.000000
75%      33538.000000
max      36612.000000
dtype: float64

In [16]:
dfgenepanel

,sample_id,dataset_title,gene_panel_size
0,NCBI568,A Spatial Transcriptomic atlas of the human ki...,36601
1,NCBI603,A new epithelial cell subpopulation predicts r...,33538
2,NCBI691,A single-cell transcriptomic analysis of endom...,36601
3,MISC32,A spatially resolved atlas of the human lung c...,33538
4,NCBI855,Batf3-dendritic cells and 4-1BB-4-1BB ligand a...,17943
...,...,...,...
83,INT24,Tertiary lymphoid structures generate and prop...,17943
84,TENX92,Visium CytAssist Gene Expression Libraries of ...,18085
85,TENX128,"Visium HD Spatial Gene Expression Library, Hum...",18085
86,TENX144,"Visium HD Spatial Gene Expression Library, Hum...",18085


In [17]:
# print the first var_name of each adata
for adata in adata_lst:
    print(adata.var_names[0])

MIR1302-2HG
MIR1302-2HG
MIR1302-2HG
MIR1302-2HG
SAMD11
OR4F5
SAMD11
MIR1302-2HG
LAMP2
RUNX1
LDHA
TCIM
EPCAM
SDC1
CTNNB1
SFRP2
IGHG1
ATP7A
A2ML1
SAMD11
MIR1302.2HG
MIR1302-2HG
MIR1302-2HG
SEC11C
SAMD11
CD19
SAMD11
MIR1302-2HG
MIR1302-2HG
SAMD11
ISG15
MIR1302-2HG
MIR1302-2HG
GNB1
MIR1302-2HG
SAMD11
PRDX4
SAMD11
DFFB
MIR1302-2HG
TNFRSF18
MIR1302-2HG
MIR1302-2HG
GPC3
SAMD11
RIDA
SAMD11
LILRA4
SAMD11
SAMD11
MIR1302-2HG
SAMD11
SAMD11
SAMD11
SAMD11
OGN
AZGP1
AZGP1
SELL
MIR1302-2HG
MIR1302-2HG
MIR1302-2HG
COL1A1
MIR1302-2HG
MIR1302-2HG
SAMD11
RHOA
STEAP4
AURKB
SAMD11
GRCh38______MIR1302-2HG
OR4F5
MIR1302-2HG
MIR1302-2HG
MIR1302-2HG
MIR1302-2HG
OR4F5
SAMD11
MIR1302-2HG
MIR1302-2HG
ENSG00000243485
SAMD11
OR4F5
SAMD11
SAMD11
SAMD11
SAMD11
MIR1302-2HG


### Gene panel filtering

Load all available h3ad files and filter genes if they are always the same expression count.

In [23]:
import logging
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gene_filter")

# 1) slice to Homo sapiens and exclude Spatial Transcriptomics
human_df = meta_df[
    (meta_df["species"] == "Homo sapiens")
    & (meta_df["st_technology"] != "Spatial Transcriptomics")
].copy()
if human_df.empty:
    raise ValueError("No datasets left after filtering by species/technology.")

# resolve ST data path
st_path = Path(cfg.st_path) if getattr(cfg, "st_path", None) else Path(cfg.data_path) / "st"

# 2) load all adatas, dropping duplicate genes
dataset_payloads = {}
for dataset_title, ids in tqdm(
    human_df.groupby("dataset_title")["id"].apply(list).items(),
    desc="Load adatas",
):
    adatas = []
    for sample_id in ids:
        adata = sc.read_h5ad(st_path / f"{sample_id}.h5ad")
        dup_mask = adata.var_names.duplicated()
        dup_count = int(dup_mask.sum())
        if dup_count:
            dup_names = adata.var_names[dup_mask][:5].tolist()
            logger.info("%s: dropping %d duplicate var_names -> %s", sample_id, dup_count, dup_names)
            adata = adata[:, ~dup_mask].copy()
        adata.obs["dataset_title"] = dataset_title
        adata.obs["sample_id"] = sample_id
        adatas.append(adata)
    dataset_payloads[dataset_title] = {"ids": ids, "adatas": adatas}

def _gene_stats(X):
    if sparse.issparse(X):
        return (
            np.asarray(X.sum(axis=0)).ravel(),
            np.asarray(X.max(axis=0).todense()).ravel(),
            np.asarray(X.min(axis=0).todense()).ravel(),
        )
    X = np.asarray(X)
    return X.sum(axis=0), X.max(axis=0), X.min(axis=0)

def _spot_stats(X):
    if sparse.issparse(X):
        if X.shape[0] == 0 or X.shape[1] == 0:
            return np.array([]), np.array([]), np.array([])
        return (
            np.asarray(X.min(axis=1)).ravel(),
            np.asarray(X.max(axis=1)).ravel(),
            np.asarray(X.mean(axis=1)).ravel(),
        )
    X = np.asarray(X)
    if X.size == 0 or X.shape[1] == 0:
        return np.array([]), np.array([]), np.array([])
    return X.min(axis=1), X.max(axis=1), X.mean(axis=1)

rng = np.random.default_rng(0)
filtered_genes = defaultdict(list)
value_summaries = {}
dropped_datasets = []

# 3) per-dataset, split by identical gene panels
for dataset_title, payload in tqdm(dataset_payloads.items(), desc="Split/filter genes"):
    ids = payload["ids"]
    adatas = payload["adatas"]

    panel_groups = {}
    for i, adata in enumerate(adatas):
        key = tuple(adata.var_names)  # order-sensitive panel signature
        panel_groups.setdefault(key, []).append(i)

    num_splits = len(panel_groups)
    pct_splits = 100.0 * num_splits / len(adatas)
    logger.info("%s: %d split(s); %.1f%% relative to %d tissues", dataset_title, num_splits, pct_splits, len(adatas))

    for split_idx, (panel_key, idxs) in enumerate(panel_groups.items(), start=1):
        split_title = f"(Split {split_idx}) {dataset_title}"
        split_ids = [ids[i] for i in idxs]
        split_adatas = [adatas[i] for i in idxs]

        common_genes = list(panel_key)  # identical by construction
        if not common_genes:
            logger.info("%s: empty gene panel -> dropping split", split_title)
            dropped_datasets.append((split_title, 100.0))
            continue

        adatas_common = [a[:, common_genes].copy() for a in split_adatas]
        concat = ad.concat(adatas_common, join="inner", keys=split_ids, label="sample_id")

        sums, maxs, mins = _gene_stats(concat.X)
        constant_mask = (maxs - mins) == 0
        genes_to_drop = concat.var_names[constant_mask].tolist()

        filtered_genes[split_title] = genes_to_drop
        drop_pct = 100 * len(genes_to_drop) / len(concat.var_names) if len(concat.var_names) else 0
        logger.info(
            "%s: drop %d genes (%d constant, %.1f%%) -> %s (max 5 shown)",
            split_title,
            len(genes_to_drop),
            int(constant_mask.sum()),
            drop_pct,
            genes_to_drop[:5],
        )

        if drop_pct > 90:
            logger.info("%s: dropping split because %.1f%% of genes would be removed", split_title, drop_pct)
            dropped_datasets.append((split_title, drop_pct))
            continue

        # per-spot stats (min/max/mean across genes)
        spot_min, spot_max, spot_mean = _spot_stats(concat.X)
        if spot_min.size == 0:
            logger.info("%s: no genes left after filtering; skipping stats", split_title)
            dropped_datasets.append((split_title, 100.0))
            continue

        spot_stats_df = pd.DataFrame(
            {"spot_min": spot_min, "spot_max": spot_max, "spot_mean": spot_mean},
            index=concat.obs_names,
        )
        logger.info(
            "%s: per-spot stats (min/max/mean across genes) describe ->\n%s",
            split_title,
            spot_stats_df.describe(),
        )

        rand_idx = int(rng.integers(0, len(split_ids)))
        rand_id = split_ids[rand_idx]
        rand_adata = split_adatas[rand_idx]
        vals = rand_adata.X.toarray() if sparse.issparse(rand_adata.X) else np.asarray(rand_adata.X)
        flat_vals = vals.ravel()
        summary = {
            "min": float(np.min(flat_vals)),
            "q1": float(np.percentile(flat_vals, 25)),
            "median": float(np.median(flat_vals)),
            "q3": float(np.percentile(flat_vals, 75)),
            "max": float(np.max(flat_vals)),
            "mean": float(np.mean(flat_vals)),
        }
        value_summaries[split_title] = {"sample_id": rand_id, "summary": summary}
        logger.info("%s: random tissue %s value summary %s", split_title, rand_id, summary)

# 4) final report of dropped datasets/splits
sep = "-" * 80
logger.info(sep)
logger.info("Dropped datasets/splits (>90%% genes removed or no genes left):")
if not dropped_datasets:
    logger.info("None")
else:
    for ds, pct in dropped_datasets:
        logger.info(" - %s: %.1f%% genes would be removed", ds, pct)
logger.info(sep)


Load adatas: 6it [00:21,  6.17s/it]/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
2026-01-19 14:55:36,353 [INFO] TENX156: dropping 3 duplicate var_names -> ['TBCE', 'HSPA14', 'TMSB15B']
/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
2026-01-19 14:55:36,974 [INFO] TENX155: dropping 3 duplicate var_names -> ['TBCE', 'HSPA14', 'TMSB15B']
/home/x_felco/workspace/Stem_stomics/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
2026-01-19 14:55:37,438 [INFO] TENX154: dropping 

In [ ]:
# generate image patch embeddings (UNI and CONCH, original and augmented)

os.makedirs(data_path / "processed_data/", exist_ok=True)
os.makedirs(data_path / "processed_data/1spot_conch_ebd/", exist_ok=True)
os.makedirs(data_path / "processed_data/1spot_uni_ebd/", exist_ok=True)
os.makedirs(data_path / "processed_data/1spot_conch_ebd_aug/", exist_ok=True)
os.makedirs(data_path / "processed_data/1spot_uni_ebd_aug/", exist_ok=True)
# if error, folder exists.

The next cell might take a long time.

In [ ]:
for i in range(len(fn_lst)):
    fn = fn_lst[i]
    adata = adata_lst[i]
    print(fn)
    image = Image.open(tif_path / f"{fn}.tif")
    get_img_patch_embd(image, adata, fn, device="cuda")#,
                    #    save_path=data_path / "processed_data/")
    print("#" * 20)

Select genes  
- Following code as an example
- Could be any given gene list saved in: data_path + "processed_data/selected_gene_list.txt"

In [33]:
union_hvg = set()

for fn_idx in range(len(fn_lst)):
    adata = adata_lst[fn_idx].copy()
    fn = fn_lst[fn_idx]
    
    # sc.pp.filter_cells(adata, min_genes=1)
    sc.pp.filter_genes(adata, min_cells=1)
    # sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000) # original=2000

    union_hvg = union_hvg.union(set(adata.var_names[adata.var["highly_variable"]]))
    print(fn, len(union_hvg))

# union_hvg = sorted([gene for gene in union_hvg if not gene.startswith(("MT", "mt", "RPS", "RPL"))]) # [optional] remove mitochondrial genes and ribosomal genes
union_hvg = sorted(list(union_hvg))
print(len(union_hvg))


/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


MEND65 2000


/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


MISC3 3628


/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


...

/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


NCBI632 5813


/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


NCBI655 6939
TENX27 7682
7682


/workspaces/docker_workspace/Stem_stomics/.venv/lib/python3.12/site-packages/pandas/core/util/hashing.py:330: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


In [34]:
# select union_hvg and concat all slides
all_count_df = pd.DataFrame(adata_lst[0][:, union_hvg].X.toarray(), 
                            columns=union_hvg, 
                            index=[fn_lst[0] + "_" + str(i) for i in range(adata_lst[0].shape[0])])

for fn_idx in range(1, len(fn_lst)):
    adata = adata_lst[fn_idx]
    df = pd.DataFrame(adata[:, union_hvg].X.toarray(), 
                      columns=union_hvg, 
                      index=[fn_lst[fn_idx] + "_" + str(i) for i in range(adata.shape[0])])
    all_count_df = pd.concat([all_count_df, df], axis=0)
    print(fn_lst[fn_idx], adata.shape, all_count_df.shape)

## print number of NaN values in the dataframe
print("Number of NaN values in the dataframe: ", all_count_df.isna().sum().sum())
all_count_df.fillna(0, inplace=True)
all_count_df = all_count_df

MISC3 (3673, 17765) (4996, 7682)
NCBI631 (2051, 17765) (7047, 7682)
NCBI632 (2743, 17765) (9790, 7682)
NCBI655 (4984, 17765) (14774, 7682)
TENX27 (4992, 17765) (19766, 7682)
Number of NaN values in the dataframe:  0


In [35]:
# order selected genes by mean and std
all_gene_order_by_mean = all_count_df.mean(axis=0).sort_values(ascending=False).index
all_gene_order_by_std = all_count_df.std(axis=0).sort_values(ascending=False).index

In [ ]:
# select top intersection of high mean and high variance genes

num_genes = 280 # to make final gene list of length 200 (original: 241)

selected_genes = sorted(list(set(all_gene_order_by_mean[:num_genes]).intersection(set(all_gene_order_by_std[:num_genes]))))
print(len(selected_genes))

200


In [44]:
with open(data_path / "processed_data/selected_gene_list.txt", "w") as f:
    for gene in selected_genes:
        f.write(gene + "\n")

with open(data_path / "processed_data/all_slide_lst.txt", "w") as f:
    for fn in fn_lst:
        f.write(fn + "\n")

# Additional Pathology Foundation Models:

H-optimus-0: https://huggingface.co/bioptimus/H-optimus-0

In [ ]:
from huggingface_hub import login
login(token="") # TODO: need to replace "" by HuggingFace Login Token

from torchvision import transforms

def get_img_patch_embd(img, 
                       adata,
                       samplename,
                       device,
                       save_path=None):

    # load model
    model = timm.create_model("hf-hub:bioptimus/H-optimus-0", pretrained=True, 
                              init_values=1e-5, dynamic_img_size=False)
    model = model.to(device)
    model = model.eval()
    transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.707223, 0.578729, 0.703617), 
        std=(0.211883, 0.230117, 0.177517)
    ),
])

    
    def get_img_embd(patch, model=model, transform=transform):
        # resize to 224 by 224
        base_width = 224
        patch_resized = patch.resize((base_width, base_width), 
                                     Image.Resampling.LANCZOS) # [224, 224]
        img_transformed = transform(transforms.ToPILImage()(torch.tensor(np.array(patch_resized)).permute(2, 0, 1))).unsqueeze(0)  # [1, 3, 224, 224]
        with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.float16):
            features = model(img_transformed.to(device))
        return torch.clone(features).detach().cpu()

    def patch_augmentation_embd(patch, num_transpose = 7):
        embd_dim = 1536
        patch_aug_embd = torch.zeros(num_transpose, embd_dim)
        for trans in range(num_transpose):
            patch_transposed = patch.transpose(trans)
            patch_embd = get_img_embd(patch_transposed)
            patch_aug_embd[trans, :] = torch.clone(patch_embd)
        return patch_aug_embd.unsqueeze(0)
    
    # process spot
    spot_diameter = adata.uns["spatial"]["ST"]["scalefactors"]["spot_diameter_fullres"]
    print("Spot diameter: ", spot_diameter)  # Spot diameter for Visium
    if spot_diameter < 224: 
        radius = 112                         # minimum patch size: 224 by 224
    else:
        radius = int(spot_diameter // 2)
    x = adata.obsm["spatial"][:, 0]          # x coordinate in H&E image
    y = adata.obsm["spatial"][:, 1]          # y coordinate in H&E image


    img_embd = None
    img_embd_aug = None
    first_patch = True

    for spot_idx in tqdm(range(len(x))):
        patch = img.crop((x[spot_idx]-radius, y[spot_idx]-radius, 
                          x[spot_idx]+radius, y[spot_idx]+radius))
        patch_embd     = get_img_embd(patch)
        patch_embd_aug = patch_augmentation_embd(patch)

        if first_patch:
            img_embd = patch_embd
            img_embd_aug = patch_embd_aug
            first_patch = False
        else:
            img_embd = torch.cat((img_embd, patch_embd), dim=0)
            img_embd_aug = torch.cat((img_embd_aug, patch_embd_aug), dim=0)

    print("Final size:", img_embd.shape, img_embd_aug.shape)

    if save_path is not None:
        torch.save(img_embd,     save_path + f"1spot_hopt0_ebd/{samplename}_hopt0.pt")
        torch.save(img_embd_aug, save_path + f"1spot_hopt0_ebd_aug/{samplename}_hopt0.pt")


Virchow2: https://huggingface.co/paige-ai/Virchow2

In [14]:
from huggingface_hub import login
login(token="") # TODO: need to replace "" by HuggingFace Login Token
import timm
import torch
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from timm.layers import SwiGLUPacked

def get_img_patch_embd(img, 
                       adata,
                       samplename,
                       device,
                       save_path=None):

    # load model
    # need to specify MLP layer and activation function for proper init
    model = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True, 
                              mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU)
    model = model.to(device)
    model = model.eval()
    transforms = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

    
    def get_img_embd(patch, model=model, transform=transforms):
        # resize to 224 by 224
        base_width = 224
        patch_resized = patch.resize((base_width, base_width), 
                                     Image.Resampling.LANCZOS) # [224, 224]
        img_transformed = transform(patch_resized).unsqueeze(0)  # [1, 3, 224, 224]
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(img_transformed.to(device))  # [1, 2560]
            class_token = output[:, 0]
            patch_tokens = output[:, 5:]
            embedding = torch.cat([class_token, patch_tokens.mean(1)], dim=-1)
        return torch.clone(embedding).detach().cpu()

    def patch_augmentation_embd(patch, num_transpose = 7):
        embd_dim = 2560
        patch_aug_embd = torch.zeros(num_transpose, embd_dim)
        for trans in range(num_transpose):
            patch_transposed = patch.transpose(trans)
            patch_embd = get_img_embd(patch_transposed)
            patch_aug_embd[trans, :] = torch.clone(patch_embd)
        return patch_aug_embd.unsqueeze(0)
    
    
    # process spot
    spot_diameter = adata.uns["spatial"]["ST"]["scalefactors"]["spot_diameter_fullres"]
    print("Spot diameter: ", spot_diameter)  # Spot diameter for Visium
    if spot_diameter < 224: 
        radius = 112                         # minimum patch size: 224 by 224
    else:
        radius = int(spot_diameter // 2)
    x = adata.obsm["spatial"][:, 0]          # x coordinate in H&E image
    y = adata.obsm["spatial"][:, 1]          # y coordinate in H&E image


    # initialize variables
    first_patch = True
    img_embd = None
    img_embd_aug = None

    for spot_idx in tqdm(range(len(x))):
        patch = img.crop((x[spot_idx]-radius, y[spot_idx]-radius, 
                          x[spot_idx]+radius, y[spot_idx]+radius))        
        patch_embd     = get_img_embd(patch)
        patch_embd_aug = patch_augmentation_embd(patch)

        if first_patch:
            img_embd = patch_embd
            img_embd_aug = patch_embd_aug
            first_patch = False
        else:
            img_embd = torch.cat((img_embd, patch_embd), dim=0)
            img_embd_aug = torch.cat((img_embd_aug, patch_embd_aug), dim=0)

    print("Final size:", img_embd.shape, img_embd_aug.shape)

    if save_path is not None:
        torch.save(img_embd,     save_path + f"1spot_virchow2_ebd/{samplename}_virchow2.pt")
        torch.save(img_embd_aug, save_path + f"1spot_virchow2_ebd_aug/{samplename}_virchow2.pt")
